In [ ]:
import os
import shutil
import random
from pathlib import Path
from collections import defaultdict

def get_class_from_label(label_path):
    """
    Read the first class ID from a YOLO label file
    Returns the class ID or None if file is empty/invalid
    """
    try:
        with open(label_path, 'r') as f:
            first_line = f.readline().strip()
            if first_line:
                class_id = int(first_line.split()[0])
                return class_id
    except:
        return None
    return None

def split_dataset_by_class(images_dir, labels_dir, output_base_dir,
                           train_per_class=40, test_per_class=10, val_per_class=10,
                           class_names=None):
    """
    Split mixed class dataset into train/test/val with equal samples per class

    Parameters:
    - images_dir: Directory containing all mixed images
    - labels_dir: Directory containing all mixed label files
    - output_base_dir: Base directory where train/test/val folders will be created
    - train_per_class: Number of samples per class for training (default: 40)
    - test_per_class: Number of samples per class for testing (default: 10)
    - val_per_class: Number of samples per class for validation (default: 10)
    - class_names: Optional dict mapping class_id to class name for display
    """

    # Create output directory structure
    splits = ['train', 'test', 'val']
    for split in splits:
        os.makedirs(os.path.join(output_base_dir, 'images', split), exist_ok=True)
        os.makedirs(os.path.join(output_base_dir, 'labels', split), exist_ok=True)

    # Get all label files
    label_files = [f for f in os.listdir(labels_dir) if f.endswith('.txt')]

    print(f"📁 Found {len(label_files)} label files")
    print("🔍 Analyzing class distribution...\n")

    # Group files by class
    class_files = defaultdict(list)

    for label_file in label_files:
        label_path = os.path.join(labels_dir, label_file)
        class_id = get_class_from_label(label_path)

        if class_id is not None:
            base_name = Path(label_file).stem
            class_files[class_id].append(base_name)

    # Display class distribution
    print("📊 Class Distribution:")
    for class_id in sorted(class_files.keys()):
        class_name = class_names.get(class_id, f"Class {class_id}") if class_names else f"Class {class_id}"
        print(f"   {class_name}: {len(class_files[class_id])} images")

    print(f"\n🎯 Target split per class:")
    print(f"   Train: {train_per_class}")
    print(f"   Test:  {test_per_class}")
    print(f"   Val:   {val_per_class}")
    print(f"   Total: {train_per_class + test_per_class + val_per_class} per class\n")

    # Split each class
    random.seed(42)  # For reproducibility

    split_counts = {'train': 0, 'test': 0, 'val': 0}

    for class_id, file_list in class_files.items():
        class_name = class_names.get(class_id, f"Class {class_id}") if class_names else f"Class {class_id}"

        # Check if we have enough samples
        required = train_per_class + test_per_class + val_per_class
        available = len(file_list)

        if available < required:
            print(f"⚠️  Warning: {class_name} has only {available} images (need {required})")
            print(f"   Using all available images with proportional split...")
            # Proportional split
            train_count = int(available * train_per_class / required)
            test_count = int(available * test_per_class / required)
            val_count = available - train_count - test_count
        else:
            train_count = train_per_class
            test_count = test_per_class
            val_count = val_per_class

        # Shuffle and split
        random.shuffle(file_list)

        train_files = file_list[:train_count]
        test_files = file_list[train_count:train_count + test_count]
        val_files = file_list[train_count + test_count:train_count + test_count + val_count]

        print(f"✅ {class_name}: Train={len(train_files)}, Test={len(test_files)}, Val={len(val_files)}")

        # Copy files to respective directories
        for split_name, split_files in [('train', train_files), ('test', test_files), ('val', val_files)]:
            for base_name in split_files:
                # Find image file with any extension
                image_copied = False
                for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
                    image_file = base_name + ext
                    src_image = os.path.join(images_dir, image_file)

                    if os.path.exists(src_image):
                        dst_image = os.path.join(output_base_dir, 'images', split_name, image_file)
                        shutil.copy2(src_image, dst_image)
                        image_copied = True
                        break

                # Copy label file
                label_file = base_name + '.txt'
                src_label = os.path.join(labels_dir, label_file)
                dst_label = os.path.join(output_base_dir, 'labels', split_name, label_file)

                if os.path.exists(src_label):
                    shutil.copy2(src_label, dst_label)
                    if image_copied:
                        split_counts[split_name] += 1

    print(f"\n🎉 Dataset split complete!")
    print(f"📊 Final counts:")
    print(f"   Train: {split_counts['train']} images")
    print(f"   Test:  {split_counts['test']} images")
    print(f"   Val:   {split_counts['val']} images")
    print(f"\n📂 Output directory: {output_base_dir}")

    return split_counts


# ==================== CONFIGURE YOUR PATHS ====================

# Input directories (mixed images and labels)
images_dir = '/content/drive/MyDrive/Major Project/Distant_images/images_ori'
labels_dir = '/content/drive/MyDrive/Major Project/Distant_images/labels_ori'

# Output directory (where train/test/val will be created)
output_base_dir = '/content/drive/MyDrive/Major Project/Distant_images'

# Class names (optional - update based on your class IDs)
# If speedbreaker=0, school_zone=1, slippery_zone=2, then:
class_names = {
    0: 'Speed Breaker',
    1: 'School Zone',
    2: 'Slippery Zone'
}

# Split configuration
train_per_class = 40
test_per_class = 10
val_per_class = 10

# ==================== RUN THE SPLIT ====================

print("🚀 Starting dataset split by class...\n")

counts = split_dataset_by_class(
    images_dir=images_dir,
    labels_dir=labels_dir,
    output_base_dir=output_base_dir,
    train_per_class=train_per_class,
    test_per_class=test_per_class,
    val_per_class=val_per_class,
    class_names=class_names
)

print("\n📊 Final Dataset Structure:")
print(f"""
{output_base_dir}/
├── images/
│   ├── train/    ({counts['train']} images)
│   ├── test/     ({counts['test']} images)
│   └── val/      ({counts['val']} images)
└── labels/
    ├── train/    ({counts['train']} labels)
    ├── test/     ({counts['test']} labels)
    └── val/      ({counts['val']} labels)
""")

print("✅ Ready for YOLO training!")
print("\n💡 Don't forget to create your data.yaml file:")
print(f"""
path: {output_base_dir}
train: images/train
val: images/val
test: images/test

nc: 3
names: ['Speed Breaker', 'School Zone', 'Slippery Zone']
""")

🚀 Starting dataset split by class...

📁 Found 181 label files
🔍 Analyzing class distribution...

📊 Class Distribution:
   Speed Breaker: 59 images
   School Zone: 60 images
   Slippery Zone: 61 images

🎯 Target split per class:
   Train: 40
   Test:  10
   Val:   10
   Total: 60 per class

⚠️  Warning: Speed Breaker has only 59 images (need 60)
   Using all available images with proportional split...
✅ Speed Breaker: Train=39, Test=9, Val=11
✅ School Zone: Train=40, Test=10, Val=10
✅ Slippery Zone: Train=40, Test=10, Val=10

🎉 Dataset split complete!
📊 Final counts:
   Train: 119 images
   Test:  29 images
   Val:   31 images

📂 Output directory: /content/drive/MyDrive/Major Project/Distant_images

📊 Final Dataset Structure:

/content/drive/MyDrive/Major Project/Distant_images/
├── images/
│   ├── train/    (119 images)
│   ├── test/     (29 images)
│   └── val/      (31 images)
└── labels/
    ├── train/    (119 labels)
    ├── test/     (29 labels)
    └── val/      (31 labels)

✅ Re

In [ ]:
import yaml

output_path = '/content/drive/MyDrive/Major Project/Distant_images/Split_dataset/data.yaml'
dataset_path = '/content/drive/MyDrive/Major Project/Distant_images/Split_dataset'

data = {
    'path': dataset_path,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 3,
    'names': ['Speed Breaker', 'School Zone', 'Slippery Zone']
}

with open(output_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False, sort_keys=False)

print(f"✅ data.yaml created at: {output_path}")

✅ data.yaml created at: /content/drive/MyDrive/Major Project/Distant_images/Split_dataset/data.yaml


In [ ]:
"""
Complete YOLOv8n Training Script for 3-Class Dataset
Trains the model and exports to TFLite and ONNX formats
"""

# ==================== INSTALL REQUIRED PACKAGES ====================
# Run this cell first if packages are not installed

!pip install ultralytics
!pip install onnx
!pip install tensorflow


# ==================== IMPORTS ====================
from ultralytics import YOLO
import os
from pathlib import Path

# ==================== CONFIGURATION ====================

# Paths
DATA_YAML = '/content/drive/MyDrive/Major Project/Distant_images/Split_dataset/data.yaml'
PROJECT_DIR = '/content/drive/MyDrive/Major Project/Distant_images/YOLO_Training'
MODEL_NAME = 'yolov8n.pt'  # Pretrained YOLOv8 nano model

# Training hyperparameters
EPOCHS = 50
IMG_SIZE = 640
BATCH_SIZE = 16  # Adjust based on your GPU memory (8, 16, or 32)
DEVICE = 0  # Use GPU 0, set to 'cpu' if no GPU

# Export settings
EXPORT_DIR = os.path.join(PROJECT_DIR, 'exported_models')

# ==================== STEP 1: INITIALIZE MODEL ====================
print("🚀 Initializing YOLOv8n model...")
model = YOLO(MODEL_NAME)

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"📊 Training will use: {'GPU' if DEVICE == 0 else 'CPU'}\n")

# ==================== STEP 2: TRAIN THE MODEL ====================
print("🏋️ Starting training...")
print(f"📁 Dataset: {DATA_YAML}")
print(f"📈 Epochs: {EPOCHS}")
print(f"📐 Image size: {IMG_SIZE}")
print(f"📦 Batch size: {BATCH_SIZE}\n")

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=PROJECT_DIR,
    name='3class_training',
    patience=10,  # Early stopping patience
    save=True,
    save_period=5,  # Save checkpoint every 10 epochs
    plots=True,  # Generate training plots
    verbose=True,

)

print("\n✅ Training completed!")

# ==================== STEP 3: EVALUATE ON TEST SET ====================
print("\n📊 Evaluating on test set...")

# Load best model
best_model_path = os.path.join(PROJECT_DIR, '3class_training', 'weights', 'best.pt')
model = YOLO(best_model_path)

# Run validation on test set
test_results = model.val(
    data=DATA_YAML,
    split='test',
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE
)

print(f"\n📈 Test Results:")
print(f"   mAP50: {test_results.results_dict['metrics/mAP50(B)']:.4f}")
print(f"   mAP50-95: {test_results.results_dict['metrics/mAP50-95(B)']:.4f}")
print(f"   Precision: {test_results.results_dict['metrics/precision(B)']:.4f}")
print(f"   Recall: {test_results.results_dict['metrics/recall(B)']:.4f}")

# ==================== STEP 4: EXPORT TO TFLITE ====================
print("\n🔄 Exporting to TFLite format...")

os.makedirs(EXPORT_DIR, exist_ok=True)

# Export to TFLite (Float32)
tflite_float32_path = model.export(
    format='tflite',
    imgsz=IMG_SIZE,
    int8=False,
    half=False
)
print(f"✅ TFLite Float32 exported: {tflite_float32_path}")

# Export to TFLite (Float16)
tflite_float16_path = model.export(
    format='tflite',
    imgsz=IMG_SIZE,
    int8=False,
    half=True
)
print(f"✅ TFLite Float16 exported: {tflite_float16_path}")

# Optional: Export to TFLite (INT8) - requires calibration data
# Uncomment if you want INT8 quantization
"""
tflite_int8_path = model.export(
    format='tflite',
    imgsz=IMG_SIZE,
    int8=True,
    data=DATA_YAML
)
print(f"✅ TFLite INT8 exported: {tflite_int8_path}")
"""

# ==================== STEP 5: EXPORT TO ONNX ====================
print("\n🔄 Exporting to ONNX format...")

onnx_path = model.export(
    format='onnx',
    imgsz=IMG_SIZE,
    dynamic=False,  # Set True for dynamic input sizes
    simplify=True,  # Simplify ONNX model
    opset=12  # ONNX opset version
)
print(f"✅ ONNX exported: {onnx_path}")

# ==================== STEP 6: COPY MODELS TO EXPORT DIRECTORY ====================
print("\n📦 Organizing exported models...")

import shutil

# Copy best.pt
best_pt_dest = os.path.join(EXPORT_DIR, 'best.pt')
shutil.copy2(best_model_path, best_pt_dest)
print(f"✅ Best PyTorch model: {best_pt_dest}")

# Copy TFLite models
if os.path.exists(tflite_float32_path):
    dest = os.path.join(EXPORT_DIR, 'best_float32.tflite')
    shutil.copy2(tflite_float32_path, dest)
    print(f"✅ TFLite Float32: {dest}")

if os.path.exists(tflite_float16_path):
    dest = os.path.join(EXPORT_DIR, 'best_float16.tflite')
    shutil.copy2(tflite_float16_path, dest)
    print(f"✅ TFLite Float16: {dest}")

# Copy ONNX model
if os.path.exists(onnx_path):
    dest = os.path.join(EXPORT_DIR, 'best.onnx')
    shutil.copy2(onnx_path, dest)
    print(f"✅ ONNX model: {dest}")

# ==================== SUMMARY ====================
print("\n" + "="*60)
print("🎉 TRAINING AND EXPORT COMPLETE!")
print("="*60)

print(f"\n📂 All files saved to:")
print(f"   Training outputs: {os.path.join(PROJECT_DIR, '3class_training')}")
print(f"   Exported models:  {EXPORT_DIR}")

print(f"\n📊 Model Performance:")
print(f"   mAP50:     {test_results.results_dict['metrics/mAP50(B)']:.4f}")
print(f"   mAP50-95:  {test_results.results_dict['metrics/mAP50-95(B)']:.4f}")
print(f"   Precision: {test_results.results_dict['metrics/precision(B)']:.4f}")
print(f"   Recall:    {test_results.results_dict['metrics/recall(B)']:.4f}")

print(f"\n📦 Exported Formats:")
print(f"   ✅ PyTorch (.pt)")
print(f"   ✅ TFLite Float32 (.tflite)")
print(f"   ✅ TFLite Float16 (.tflite)")
print(f"   ✅ ONNX (.onnx)")

print(f"\n💡 For Raspberry Pi 4 deployment:")
print(f"   Use: best_float16.tflite (recommended)")
print(f"   Location: {os.path.join(EXPORT_DIR, 'best_float16.tflite')}")

print("\n✅ Ready for deployment!")


# ==================== OPTIONAL: TEST INFERENCE ====================
"""
# Uncomment to test inference on a sample image

test_image = '/content/drive/MyDrive/Major Project/Split_Dataset/images/test/IMG_20260203_183214.jpg'

print(f"\n🔍 Testing inference on sample image...")
results = model.predict(
    source=test_image,
    imgsz=IMG_SIZE,
    conf=0.25,  # Confidence threshold
    save=True,
    save_dir=os.path.join(EXPORT_DIR, 'predictions')
)

print(f"✅ Prediction saved to: {os.path.join(EXPORT_DIR, 'predictions')}")
"""

🚀 Initializing YOLOv8n model...
✅ Model loaded: yolov8n.pt
📊 Training will use: GPU

🏋️ Starting training...
📁 Dataset: /content/drive/MyDrive/Major Project/Distant_images/Split_dataset/data.yaml
📈 Epochs: 50
📐 Image size: 640
📦 Batch size: 16

Ultralytics 8.4.12 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Major Project/Distant_images/Split_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras

'\n# Uncomment to test inference on a sample image\n\ntest_image = \'/content/drive/MyDrive/Major Project/Split_Dataset/images/test/IMG_20260203_183214.jpg\'\n\nprint(f"\n🔍 Testing inference on sample image...")\nresults = model.predict(\n    source=test_image,\n    imgsz=IMG_SIZE,\n    conf=0.25,  # Confidence threshold\n    save=True,\n    save_dir=os.path.join(EXPORT_DIR, \'predictions\')\n)\n\nprint(f"✅ Prediction saved to: {os.path.join(EXPORT_DIR, \'predictions\')}")\n'